# 05a — Quantitative error analysis (Week 5 Phase 1)

Pure analysis of the three frozen TEST prediction files from Weeks 3-4. No
model runs, no test-set changes — just slicing the existing artefacts to
understand *where* and *why* the champion (PhoBERT, test F1_macro 0.6618)
still fails.

Inputs (all row-aligned to `data/processed/test_cleaned.csv`):

* `results/test_predictions.csv`            — LR + char (Week 3)
* `results/bilstm_v2_test_predictions.csv`  — BiLSTM v2 (Week 4)
* `results/phobert_test_predictions.csv`    — PhoBERT (Week 4, champion)

Parts:
1. **A** — load + align the 3 files into one master frame.
2. **B** — cross-model disagreement sets (who's right when, and on what).
3. **C** — PhoBERT confidence calibration (reliability diagram, high-conf wrongs).
4. **D** — length stratification (does short text break the models?).
5. **E** — OFFENSIVE-class deep dive (the persistent failure).
6. **F** — synthesis → `report/week5_quantitative_findings.md`.

Outputs:

* `results/test_master_predictions.csv`
* `results/disagreement_set_{A,B,C,D,E}.csv`
* `results/high_conf_wrong.csv`
* `results/offensive_failure_analysis.csv`
* `results/figures/27_disagreement_classes.png`
* `results/figures/28_phobert_confidence.png`
* `results/figures/29_calibration.png`
* `results/figures/30_length_stratified_f1.png`
* `results/figures/31_offensive_failures.png`
* `report/week5_quantitative_findings.md`


In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

import sys
for _k in [k for k in sys.modules if k == "src" or k.startswith("src.")]:
    del sys.modules[_k]

import json, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path = [str(ROOT)] + [p for p in sys.path if p != str(ROOT)]

from configs.config import COLUMNS, LABEL_MAP

RESULTS_DIR = ROOT / "results"
FIG_DIR     = RESULTS_DIR / "figures"
REPORT_DIR  = ROOT / "report"
PROCESSED_DIR = ROOT / "data" / "processed"
for d in (FIG_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
LABEL_NAMES  = [LABEL_MAP[i] for i in (0, 1, 2)]   # CLEAN / OFFENSIVE / HATE
LABEL_COLORS = {0: "#4CAF50", 1: "#FF9800", 2: "#F44336"}
print("Setup done.")


## A. Load + align the 3 prediction files

The three files use slightly different schemas (LR from Week 3 uses
`predicted_label` / `proba_*`; the Week-4 files use `pred_label` /
`prob_*`). We normalise column names, then **assert** the `true_label`
sequence is identical across all three — that proves they're row-aligned
to the same test split before we join by position.

In [ ]:
def _pred_col(df):
    for c in ("pred_label", "predicted_label"):
        if c in df.columns:
            return c
    raise KeyError(f"no prediction column in {list(df.columns)}")

def _prob_cols(df):
    if "prob_clean" in df.columns:
        return ["prob_clean", "prob_offensive", "prob_hate"]
    if "proba_clean" in df.columns:
        return ["proba_clean", "proba_offensive", "proba_hate"]
    raise KeyError(f"no probability columns in {list(df.columns)}")

def _text_col(df):
    for c in ("text_original", "text"):
        if c in df.columns:
            return c
    raise KeyError("no text column")

lr_df  = pd.read_csv(RESULTS_DIR / "test_predictions.csv")
bl_df  = pd.read_csv(RESULTS_DIR / "bilstm_v2_test_predictions.csv")
phb_df = pd.read_csv(RESULTS_DIR / "phobert_test_predictions.csv")
print(f"LR     : {len(lr_df):,} rows  cols={list(lr_df.columns)}")
print(f"BiLSTM : {len(bl_df):,} rows  cols={list(bl_df.columns)}")
print(f"PhoBERT: {len(phb_df):,} rows  cols={list(phb_df.columns)}")

# Same length?
n = len(phb_df)
assert len(lr_df) == n == len(bl_df), "row-count mismatch across prediction files"

# Same true-label sequence → proves row alignment (same test order).
lr_true  = lr_df["true_label"].to_numpy()
bl_true  = bl_df["true_label"].to_numpy()
phb_true = phb_df["true_label"].to_numpy()
assert np.array_equal(lr_true, phb_true) and np.array_equal(bl_true, phb_true), (
    "true_label sequences differ across files — the CSVs are NOT row-aligned. "
    "Re-export them from the same test split before joining."
)
print(f"\n✓ all 3 files row-aligned on true_label ({n:,} samples)")

# Cleaned text for length; fall back to raw test_cleaned.csv if BiLSTM/PhoBERT
# lack the cleaned column.
if "text_cleaned" in phb_df.columns:
    text_cleaned = phb_df["text_cleaned"].fillna("").astype(str)
else:
    _tc = pd.read_csv(PROCESSED_DIR / "test_cleaned.csv")
    text_cleaned = _tc["cleaned"].fillna("").astype(str).reset_index(drop=True)

phb_probs = phb_df[_prob_cols(phb_df)].to_numpy()

master = pd.DataFrame({
    "text":          phb_df[_text_col(phb_df)].astype(str),
    "text_cleaned":  text_cleaned.values,
    "length":        text_cleaned.str.split().apply(len).values,
    "true_label":    phb_true,
    "true_name":     [LABEL_MAP[int(l)] for l in phb_true],
    "lr_pred":       lr_df[_pred_col(lr_df)].to_numpy(),
    "lr_conf":       lr_df["confidence"].to_numpy() if "confidence" in lr_df.columns else np.nan,
    "bilstm_pred":   bl_df[_pred_col(bl_df)].to_numpy(),
    "bilstm_conf":   bl_df["confidence"].to_numpy() if "confidence" in bl_df.columns else np.nan,
    "phobert_pred":  phb_df[_pred_col(phb_df)].to_numpy(),
    "phobert_conf":  phb_df["confidence"].to_numpy() if "confidence" in phb_df.columns else phb_probs.max(axis=1),
    "prob_clean":    phb_probs[:, 0],
    "prob_off":      phb_probs[:, 1],
    "prob_hate":     phb_probs[:, 2],
})

# Correctness flags per model.
master["lr_correct"]      = master["lr_pred"]      == master["true_label"]
master["bilstm_correct"]  = master["bilstm_pred"]  == master["true_label"]
master["phobert_correct"] = master["phobert_pred"] == master["true_label"]

print(f"\nmaster shape: {master.shape}")
print(f"true_label dist: {master['true_label'].value_counts().sort_index().to_dict()}")
print(f"per-model accuracy: LR={master['lr_correct'].mean():.4f}  "
      f"BiLSTM={master['bilstm_correct'].mean():.4f}  "
      f"PhoBERT={master['phobert_correct'].mean():.4f}")
master.head(3)


In [ ]:
master_path = RESULTS_DIR / "test_master_predictions.csv"
master.to_csv(master_path, index=False)
print(f"✓ saved → {master_path}  ({len(master):,} rows × {master.shape[1]} cols)")


## B. Cross-model disagreement sets

Five subsets that localise *where* the models diverge. The interesting
science is in A (LR wins) and B (PhoBERT wins) — they reveal what each
inductive bias captures — and D (everyone fails), which is the
annotation-ambiguity / out-of-distribution candidate pool.

In [ ]:
lr_ok  = master["lr_correct"].to_numpy()
bl_ok  = master["bilstm_correct"].to_numpy()
phb_ok = master["phobert_correct"].to_numpy()

subsets = {
    "A_LRright_PhoBERTwrong":  lr_ok & (~phb_ok),
    "B_PhoBERTright_LRwrong":  phb_ok & (~lr_ok),
    "C_BiLSTMright_PhoBERTwrong": bl_ok & (~phb_ok),
    "D_all_wrong":             (~lr_ok) & (~bl_ok) & (~phb_ok),
    "E_all_right":             lr_ok & bl_ok & phb_ok,
}

print(f"{'subset':<28s} {'size':>6s} {'% total':>8s}")
print("-" * 46)
for name, mask in subsets.items():
    print(f"{name:<28s} {int(mask.sum()):>6d} {100*mask.mean():>7.2f}%")


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(texts, ngram=(2, 2), k=8):
    """Most frequent n-grams in a list of cleaned strings."""
    texts = [t for t in texts if isinstance(t, str) and t.strip()]
    if len(texts) < 2:
        return []
    try:
        vec = CountVectorizer(ngram_range=ngram, min_df=2, token_pattern=r"(?u)\b\w[\w_]+\b")
        X = vec.fit_transform(texts)
        freqs = np.asarray(X.sum(axis=0)).ravel()
        vocab = np.array(vec.get_feature_names_out())
        order = freqs.argsort()[::-1][:k]
        return [(vocab[i], int(freqs[i])) for i in order]
    except ValueError:
        return []

subset_stats = {}
for name, mask in subsets.items():
    sub = master[mask]
    stats = {
        "size":       int(mask.sum()),
        "pct_total":  float(100 * mask.mean()),
        "class_dist": sub["true_label"].value_counts().sort_index().to_dict(),
        "len_mean":   float(sub["length"].mean()) if len(sub) else 0.0,
        "len_std":    float(sub["length"].std()) if len(sub) else 0.0,
        "top_bigrams": top_ngrams(sub["text_cleaned"].tolist(), (2, 2), 8),
    }
    subset_stats[name] = stats

    # Persist each subset for qualitative inspection (Day 2).
    cols = ["text", "text_cleaned", "length", "true_label", "true_name",
            "lr_pred", "bilstm_pred", "phobert_pred", "phobert_conf",
            "prob_clean", "prob_off", "prob_hate"]
    sub[cols].to_csv(RESULTS_DIR / f"disagreement_set_{name}.csv", index=False)

# Pretty print.
for name, s in subset_stats.items():
    dist_named = {LABEL_MAP[k]: v for k, v in s["class_dist"].items()}
    print(f"\n■ {name}  (n={s['size']}, {s['pct_total']:.1f}%)")
    print(f"   class dist : {dist_named}")
    print(f"   length     : mean={s['len_mean']:.1f}  std={s['len_std']:.1f}")
    if s["top_bigrams"]:
        print(f"   top bigrams: {', '.join(f'{w}({c})' for w, c in s['top_bigrams'][:6])}")
print(f"\n✓ saved 5 subset CSVs → results/disagreement_set_*.csv")


In [ ]:
# Stacked bar: true-class composition of each disagreement subset.
fig, ax = plt.subplots(figsize=(10, 5.5))

names = list(subsets.keys())
bottoms = np.zeros(len(names))
for cls in (0, 1, 2):
    vals = np.array([subset_stats[n]["class_dist"].get(cls, 0) for n in names], dtype=float)
    ax.bar(names, vals, bottom=bottoms, label=LABEL_MAP[cls], color=LABEL_COLORS[cls])
    bottoms += vals

ax.set_ylabel("Sample count")
ax.set_title("True-class composition of cross-model disagreement subsets")
ax.legend(title="True label")
ax.set_xticklabels(names, rotation=20, ha="right", fontsize=9)
for i, n in enumerate(names):
    ax.text(i, bottoms[i] + max(bottoms)*0.01, str(int(bottoms[i])), ha="center", fontsize=9)
plt.tight_layout()
fig_path = FIG_DIR / "27_disagreement_classes.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"✓ saved → {fig_path}")


In [ ]:
# 10 random examples per subset (focus on the analytically interesting ones).
pd.set_option("display.max_colwidth", 70)
for name in ["A_LRright_PhoBERTwrong", "B_PhoBERTright_LRwrong", "D_all_wrong"]:
    sub = master[subsets[name]]
    k = min(10, len(sub))
    sample = sub.sample(k, random_state=RANDOM_STATE) if k else sub
    print(f"\n════ {name}  (showing {k} of {len(sub)}) ════")
    view = sample[["text", "true_name", "lr_pred", "bilstm_pred", "phobert_pred"]].copy()
    view["lr_pred"]      = view["lr_pred"].map(LABEL_MAP)
    view["bilstm_pred"]  = view["bilstm_pred"].map(LABEL_MAP)
    view["phobert_pred"] = view["phobert_pred"].map(LABEL_MAP)
    print(view.to_string(index=False))
pd.reset_option("display.max_colwidth")


## C. PhoBERT confidence calibration

Is PhoBERT's softmax probability a trustworthy confidence? A
reliability diagram answers it: points below the diagonal = over-confident
(claims more certainty than its accuracy warrants).

In [ ]:
# Dev predictions (from 04c) for the dev-vs-test confidence comparison.
dev_csv = RESULTS_DIR / "phobert_dev_predictions.csv"
have_dev = dev_csv.exists()
if have_dev:
    dev_df = pd.read_csv(dev_csv)
    dev_conf = dev_df["confidence"].to_numpy()
    dev_correct = (dev_df["predicted_label"].to_numpy() == dev_df["true_label"].to_numpy())

test_conf    = master["phobert_conf"].to_numpy()
test_correct = phb_ok

fig, axes = plt.subplots(1, 2 if have_dev else 1, figsize=(13 if have_dev else 7, 4.5), squeeze=False)

def _conf_hist(ax, conf, correct, title):
    bins = np.linspace(0.33, 1.0, 28)
    ax.hist(conf[correct],  bins=bins, alpha=0.7, label="correct", color="#54A24B")
    ax.hist(conf[~correct], bins=bins, alpha=0.7, label="wrong",   color="#E45756")
    ax.set_xlabel("PhoBERT max softmax (confidence)")
    ax.set_ylabel("count")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

_conf_hist(axes[0][0], test_conf, test_correct, "TEST — confidence by correctness")
if have_dev:
    _conf_hist(axes[0][1], dev_conf, dev_correct, "DEV — confidence by correctness")

plt.tight_layout()
fig_path = FIG_DIR / "28_phobert_confidence.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"✓ saved → {fig_path}")
print(f"TEST mean confidence: correct={test_conf[test_correct].mean():.3f}  "
      f"wrong={test_conf[~test_correct].mean():.3f}")


In [ ]:
# Reliability diagram: bin by confidence decile, compare mean-confidence vs accuracy.
bins = np.linspace(0.0, 1.0, 11)
bin_idx = np.digitize(test_conf, bins) - 1
bin_idx = np.clip(bin_idx, 0, 9)

rows = []
for b in range(10):
    mask = bin_idx == b
    if mask.sum() == 0:
        continue
    rows.append({
        "bin":       f"[{bins[b]:.1f}-{bins[b+1]:.1f}]",
        "count":     int(mask.sum()),
        "avg_conf":  float(test_conf[mask].mean()),
        "accuracy":  float(test_correct[mask].mean()),
    })
calib_df = pd.DataFrame(rows)
print(calib_df.to_string(index=False, formatters={
    "avg_conf": "{:.3f}".format, "accuracy": "{:.3f}".format}))

# Expected Calibration Error (sample-weighted gap).
ece = float(np.sum(calib_df["count"] * np.abs(calib_df["avg_conf"] - calib_df["accuracy"]))
            / calib_df["count"].sum())

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 1], [0, 1], "--", color="#888", label="perfect calibration")
ax.plot(calib_df["avg_conf"], calib_df["accuracy"], "o-", color="#4C78A8", label="PhoBERT")
for _, r in calib_df.iterrows():
    ax.annotate(f"n={r['count']}", (r["avg_conf"], r["accuracy"]),
                xytext=(5, -10), textcoords="offset points", fontsize=8)
ax.fill_between([0, 1], [0, 1], [0, 0], alpha=0.05, color="red")
ax.set_xlabel("Mean predicted confidence (per bin)")
ax.set_ylabel("Empirical accuracy (per bin)")
ax.set_title(f"PhoBERT reliability diagram — TEST  (ECE = {ece:.4f})")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
ax.text(0.55, 0.10, "points below diagonal\n= over-confident",
        fontsize=9, color="#B22222")
plt.tight_layout()
fig_path = FIG_DIR / "29_calibration.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"\n✓ saved → {fig_path}")
print(f"Expected Calibration Error (ECE): {ece:.4f}")
overconf = (calib_df["avg_conf"] > calib_df["accuracy"]).mean()
print(f"Bins over-confident: {overconf*100:.0f}%  "
      f"→ {'over-confident overall' if overconf > 0.5 else 'reasonably calibrated'}")


In [ ]:
# High-confidence wrong predictions: where the model is loudly mistaken.
hcw = master[(master["phobert_conf"] > 0.8) & (~master["phobert_correct"])].copy()
hcw = hcw.sort_values("phobert_conf", ascending=False)

print(f"High-confidence (>0.8) wrong PhoBERT predictions: {len(hcw):,} "
      f"({100*len(hcw)/len(master):.2f}% of test)\n")

# Confusion breakdown of the high-conf wrongs.
hcw_conf = hcw.groupby(["true_name", hcw["phobert_pred"].map(LABEL_MAP)]).size()
print("High-conf-wrong direction (true → pred):")
print(hcw_conf.to_string())

cols = ["text", "true_name", "phobert_pred", "phobert_conf", "lr_pred", "bilstm_pred"]
top20 = hcw[cols].head(20).copy()
top20["phobert_pred"] = top20["phobert_pred"].map(LABEL_MAP)
top20["lr_pred"]      = top20["lr_pred"].map(LABEL_MAP)
top20["bilstm_pred"]  = top20["bilstm_pred"].map(LABEL_MAP)
pd.set_option("display.max_colwidth", 60)
print(f"\nTop 20 most-confident wrongs:")
print(top20.to_string(index=False))
pd.reset_option("display.max_colwidth")

hcw.to_csv(RESULTS_DIR / "high_conf_wrong.csv", index=False)
print(f"\n✓ saved → {RESULTS_DIR / 'high_conf_wrong.csv'}")


## D. Length stratification

Does performance collapse on very short comments (the ones we filtered out
of training)? Five length buckets, F1_macro per model per bucket.

In [ ]:
def length_bucket(n):
    if n <= 3:  return "1. very short [1-3]"
    if n <= 7:  return "2. short [4-7]"
    if n <= 15: return "3. medium [8-15]"
    if n <= 30: return "4. long [16-30]"
    return "5. very long [31+]"

master["len_bucket"] = master["length"].apply(length_bucket)
bucket_order = ["1. very short [1-3]", "2. short [4-7]", "3. medium [8-15]",
                "4. long [16-30]", "5. very long [31+]"]

def _f1_macro(sub, pred_col):
    if len(sub) == 0:
        return np.nan
    return f1_score(sub["true_label"], sub[pred_col], labels=[0, 1, 2],
                    average="macro", zero_division=0)

def _f1_per_class(sub, pred_col):
    if len(sub) == 0:
        return [np.nan, np.nan, np.nan]
    return f1_score(sub["true_label"], sub[pred_col], labels=[0, 1, 2],
                    average=None, zero_division=0).tolist()

bucket_rows = []
for b in bucket_order:
    sub = master[master["len_bucket"] == b]
    phb_pc = _f1_per_class(sub, "phobert_pred")
    bucket_rows.append({
        "bucket":       b,
        "count":        len(sub),
        "pct":          100 * len(sub) / len(master),
        "f1_lr":        _f1_macro(sub, "lr_pred"),
        "f1_bilstm":    _f1_macro(sub, "bilstm_pred"),
        "f1_phobert":   _f1_macro(sub, "phobert_pred"),
        "phb_f1_clean": phb_pc[0],
        "phb_f1_off":   phb_pc[1],
        "phb_f1_hate":  phb_pc[2],
    })
bucket_df = pd.DataFrame(bucket_rows)
print(bucket_df.to_string(index=False, formatters={
    "pct":          "{:.1f}".format,
    "f1_lr":        "{:.4f}".format,
    "f1_bilstm":    "{:.4f}".format,
    "f1_phobert":   "{:.4f}".format,
    "phb_f1_clean": "{:.4f}".format,
    "phb_f1_off":   "{:.4f}".format,
    "phb_f1_hate":  "{:.4f}".format,
}))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
short_labels = [b.split(". ")[1] for b in bucket_order]

# (left) grouped bar: F1_macro per bucket per model
ax = axes[0]
x = np.arange(len(bucket_order)); w = 0.27
for i, (col, name, color) in enumerate([
    ("f1_lr", "LR", "#9D7BBA"), ("f1_bilstm", "BiLSTM v2", "#4C78A8"), ("f1_phobert", "PhoBERT", "#54A24B"),
]):
    ax.bar(x + (i-1)*w, bucket_df[col], w, label=name, color=color)
ax.set_xticks(x); ax.set_xticklabels(short_labels, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("F1 macro"); ax.set_title("F1 macro by length bucket")
ax.legend(); ax.grid(alpha=0.3, axis="y")
for i, c in enumerate(bucket_df["count"]):
    ax.text(i, 0.02, f"n={c}", ha="center", fontsize=8, color="#333")

# (right) PhoBERT per-class F1 vs length
ax = axes[1]
for col, name, color in [("phb_f1_clean","CLEAN","#4CAF50"),
                         ("phb_f1_off","OFFENSIVE","#FF9800"),
                         ("phb_f1_hate","HATE","#F44336")]:
    ax.plot(short_labels, bucket_df[col], "o-", label=name, color=color)
ax.set_ylabel("PhoBERT F1"); ax.set_title("PhoBERT per-class F1 by length")
ax.set_xticklabels(short_labels, rotation=15, ha="right", fontsize=9)
ax.legend(); ax.grid(alpha=0.3)

fig_path = FIG_DIR / "30_length_stratified_f1.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"✓ saved → {fig_path}")

worst = bucket_df.loc[bucket_df["f1_phobert"].idxmin()]
print(f"\nPhoBERT worst bucket: {worst['bucket']}  (F1_macro={worst['f1_phobert']:.4f}, n={int(worst['count'])})")


## E. OFFENSIVE-class deep dive

OFFENSIVE is the worst class for every model. Break down exactly where the
444 true-OFFENSIVE test samples go, compare dev vs test, and inspect the
vocabulary of the OFFENSIVE→CLEAN and OFFENSIVE→HATE failures.

In [ ]:
off = master[master["true_label"] == 1]
n_off = len(off)
phb_tp     = int((off["phobert_pred"] == 1).sum())
phb_to_cln = int((off["phobert_pred"] == 0).sum())
phb_to_hat = int((off["phobert_pred"] == 2).sum())

print(f"PhoBERT — true OFFENSIVE breakdown (TEST, n={n_off}):")
print(f"  → OFFENSIVE (TP) : {phb_tp:4d}  ({100*phb_tp/n_off:.1f}%)")
print(f"  → CLEAN (FN)     : {phb_to_cln:4d}  ({100*phb_to_cln/n_off:.1f}%)")
print(f"  → HATE (FN)      : {phb_to_hat:4d}  ({100*phb_to_hat/n_off:.1f}%)")

# Dev vs test OFFENSIVE recall (→ confirms the dev-test gap on this class).
print(f"\nOFFENSIVE recall dev vs test:")
test_off_recall = phb_tp / n_off
if have_dev:
    dev_off = dev_df[dev_df["true_label"] == 1]
    dev_off_recall = (dev_off["predicted_label"] == 1).mean()
    print(f"  dev  : {dev_off_recall:.4f}  (n={len(dev_off)})")
print(f"  test : {test_off_recall:.4f}  (n={n_off})")
if have_dev:
    print(f"  Δ    : {test_off_recall - dev_off_recall:+.4f}  "
          f"→ {'confirms dev→test degradation' if test_off_recall < dev_off_recall else 'stable'}")


In [ ]:
# Vocabulary of OFFENSIVE failures — what words push the model to CLEAN vs HATE?
def word_counts(texts, k=15):
    c = Counter()
    for t in texts:
        if isinstance(t, str):
            c.update(t.split())
    return c.most_common(k)

off_to_clean = off[off["phobert_pred"] == 0]["text_cleaned"]
off_to_hate  = off[off["phobert_pred"] == 2]["text_cleaned"]
off_correct  = off[off["phobert_pred"] == 1]["text_cleaned"]

wc_clean = word_counts(off_to_clean)
wc_hate  = word_counts(off_to_hate)
wc_tp    = word_counts(off_correct)

print("Top words — OFFENSIVE predicted CLEAN (under-flagged):")
print("  " + ", ".join(f"{w}({c})" for w, c in wc_clean))
print("\nTop words — OFFENSIVE predicted HATE (over-escalated):")
print("  " + ", ".join(f"{w}({c})" for w, c in wc_hate))
print("\nTop words — OFFENSIVE correctly flagged:")
print("  " + ", ".join(f"{w}({c})" for w, c in wc_tp))

# Overlap diagnostics.
set_clean = {w for w, _ in wc_clean}
set_hate  = {w for w, _ in wc_hate}
set_tp    = {w for w, _ in wc_tp}
print(f"\nVocab overlap (top-15): FN-CLEAN ∩ correct = {len(set_clean & set_tp)}  "
      f"FN-HATE ∩ correct = {len(set_hate & set_tp)}")

# Save the OFFENSIVE failure rows for qualitative review.
off_fail = off[off["phobert_pred"] != 1].copy()
off_fail["phobert_pred_name"] = off_fail["phobert_pred"].map(LABEL_MAP)
off_fail[["text", "text_cleaned", "length", "phobert_pred_name", "phobert_conf",
          "prob_clean", "prob_off", "prob_hate", "lr_pred", "bilstm_pred"]].to_csv(
    RESULTS_DIR / "offensive_failure_analysis.csv", index=False)
print(f"\n✓ saved → {RESULTS_DIR / 'offensive_failure_analysis.csv'}  ({len(off_fail)} rows)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

# (left) where do true-OFFENSIVE samples go (PhoBERT)
ax = axes[0]
parts = [phb_tp, phb_to_cln, phb_to_hat]
labels = [f"→OFFENSIVE\n(correct)\n{phb_tp}",
          f"→CLEAN\n(under-flag)\n{phb_to_cln}",
          f"→HATE\n(over-escalate)\n{phb_to_hat}"]
colors = ["#FF9800", "#4CAF50", "#F44336"]
ax.bar(range(3), parts, color=colors)
ax.set_xticks(range(3)); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("count"); ax.set_title(f"True OFFENSIVE routing — PhoBERT (n={n_off})")
for i, v in enumerate(parts):
    ax.text(i, v + 3, f"{100*v/n_off:.1f}%", ha="center", fontsize=10)
ax.grid(alpha=0.3, axis="y")

# (right) top words in the two failure directions
ax = axes[1]
top_clean = wc_clean[:8][::-1]
y = np.arange(len(top_clean))
ax.barh(y, [c for _, c in top_clean], color="#4CAF50", alpha=0.8)
ax.set_yticks(y); ax.set_yticklabels([w for w, _ in top_clean], fontsize=9)
ax.set_xlabel("frequency"); ax.set_title("Top words: OFFENSIVE → CLEAN failures")
ax.grid(alpha=0.3, axis="x")

fig_path = FIG_DIR / "31_offensive_failures.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"✓ saved → {fig_path}")


## F. Synthesis → `report/week5_quantitative_findings.md`

Auto-generate the Phase-1 findings doc from the numbers computed above,
including the tagging plan for the Day-2 qualitative pass.

In [ ]:
lines = []
lines.append("# Week 5 Phase 1: Quantitative Error Analysis")
lines.append("")
lines.append("_Auto-generated by `notebooks/05a_quantitative_errors.ipynb`. "
             "Analysis only — no predictions modified._")
lines.append("")

# --- Disagreement insights
lines.append("## Disagreement insights")
lines.append("")
lines.append("| Subset | Size | % test | Dominant true class |")
lines.append("|---|---|---|---|")
for name, s in subset_stats.items():
    if s["class_dist"]:
        dom = max(s["class_dist"].items(), key=lambda kv: kv[1])
        dom_str = f"{LABEL_MAP[dom[0]]} ({dom[1]})"
    else:
        dom_str = "—"
    lines.append(f"| {name} | {s['size']} | {s['pct_total']:.1f}% | {dom_str} |")
lines.append("")
A = subset_stats["A_LRright_PhoBERTwrong"]; B = subset_stats["B_PhoBERTright_LRwrong"]; D = subset_stats["D_all_wrong"]
lines.append(f"- **PhoBERT's net win** over LR: B−A = {B['size']}−{A['size']} = "
             f"{B['size']-A['size']} samples ({(B['size']-A['size'])/len(master)*100:+.1f}% of test).")
lines.append(f"- **Universal failures (D)**: {D['size']} samples ({D['pct_total']:.1f}%) "
             f"wrong for all 3 architectures — the annotation-ambiguity / OOD candidate pool.")
lines.append("")

# --- Confidence calibration
lines.append("## Confidence calibration")
lines.append("")
lines.append(f"- Expected Calibration Error (ECE) on TEST: **{ece:.4f}**.")
lines.append(f"- Mean confidence: correct = {test_conf[test_correct].mean():.3f}, "
             f"wrong = {test_conf[~test_correct].mean():.3f}.")
lines.append(f"- High-confidence (>0.8) wrong predictions: **{len(hcw)}** "
             f"({100*len(hcw)/len(master):.2f}% of test).")
_verdict = "over-confident" if (calib_df['avg_conf'] > calib_df['accuracy']).mean() > 0.5 else "reasonably calibrated"
lines.append(f"- Verdict: PhoBERT is **{_verdict}** (majority of confidence bins sit "
             f"{'above' if _verdict=='over-confident' else 'near'} the diagonal).")
lines.append("")

# --- Length
lines.append("## Length-dependent performance")
lines.append("")
lines.append("| Bucket | n | F1 LR | F1 BiLSTM | F1 PhoBERT |")
lines.append("|---|---|---|---|---|")
for _, r in bucket_df.iterrows():
    lines.append(f"| {r['bucket']} | {int(r['count'])} | {r['f1_lr']:.4f} "
                 f"| {r['f1_bilstm']:.4f} | {r['f1_phobert']:.4f} |")
lines.append("")
_worst = bucket_df.loc[bucket_df["f1_phobert"].idxmin()]
lines.append(f"- PhoBERT's weakest bucket: **{_worst['bucket']}** "
             f"(F1_macro = {_worst['f1_phobert']:.4f}, n = {int(_worst['count'])}).")
lines.append(f"- Confirms **H2**: short comments are genuinely harder — they were "
             f"filtered out of training (`min_length=3`) but kept in test.")
lines.append("")

# --- OFFENSIVE
lines.append("## OFFENSIVE class deep dive")
lines.append("")
lines.append(f"- True OFFENSIVE on TEST: {n_off} samples.")
lines.append(f"  - correctly flagged: {phb_tp} ({100*phb_tp/n_off:.1f}%)")
lines.append(f"  - → CLEAN (under-flagged): {phb_to_cln} ({100*phb_to_cln/n_off:.1f}%)")
lines.append(f"  - → HATE (over-escalated): {phb_to_hat} ({100*phb_to_hat/n_off:.1f}%)")
if have_dev:
    lines.append(f"- OFFENSIVE recall: dev {dev_off_recall:.4f} → test {test_off_recall:.4f} "
                 f"({test_off_recall-dev_off_recall:+.4f}).")
lines.append(f"- Top OFFENSIVE→CLEAN words: {', '.join(w for w, _ in wc_clean[:8])}.")
lines.append("")

# --- Next steps
lines.append("## Next steps (Day 2 — Qualitative)")
lines.append("")
lines.append("Manually tag a stratified sample (~80-100 cases) drawn from the saved "
             "disagreement CSVs, using this schema:")
lines.append("")
lines.append("| Category | Definition |")
lines.append("|---|---|")
lines.append("| sarcasm/irony      | surface-positive, intended-negative |")
lines.append("| code-switching     | mixes English/Vietnamese, esp. slurs |")
lines.append("| teencode/obfuscation | `vcl`, `vl`, `dm`, leetspeak, spacing tricks |")
lines.append("| implicit/contextual | toxic only given world knowledge / target |")
lines.append("| annotation-doubt   | label looks wrong or genuinely borderline |")
lines.append("| short/low-context  | ≤3 tokens, not enough signal |")
lines.append("")
lines.append("Sampling plan:")
lines.append(f"- ~30 from `disagreement_set_D_all_wrong.csv` (universal failures, n={D['size']})")
lines.append(f"- ~25 from `disagreement_set_A_LRright_PhoBERTwrong.csv` (PhoBERT regressions, n={A['size']})")
lines.append(f"- ~25 from `disagreement_set_B_PhoBERTright_LRwrong.csv` (PhoBERT wins, n={B['size']})")
lines.append(f"- ~20 from `high_conf_wrong.csv` (confident mistakes, n={len(hcw)})")
lines.append("- All `random_state=42`; tag in a spreadsheet, then re-import for Phase-2 stats.")

out = REPORT_DIR / "week5_quantitative_findings.md"
out.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"✓ saved → {out}\n")
print(out.read_text(encoding="utf-8"))
